In [ ]:
# ==========================================
# Problem 2: Transformer From Scratch
# Notebook: problem2.ipynb
# ==========================================

import os
import time
import math
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

# Directory and outputs configuration
OS_FIGURES_DIR = "figures"
os.makedirs(OS_FIGURES_DIR, exist_ok=True)
RESULTS_PATH = "results.csv"
SAMPLES_PATH = "generated_samples.txt"

def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def log_result(data_dict):
    columns = [
        "problem", "model", "seed", "seq_len", "test_acc", "macro_f1",
        "test_mse", "params", "sec_per_epoch", "val_loss_nats", "bits_per_char", "notes"
    ]
    row = {col: data_dict.get(col, "") for col in columns}
    df = pd.DataFrame([row])
    header = not os.path.exists(RESULTS_PATH)
    df.to_csv(RESULTS_PATH, mode="a", index=False, header=header)

# ==========================================
# Data Helper: UCI HAR Data Dataloaders
# ==========================================
def download_har_if_needed():
    url = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
    if not os.path.exists('har.zip'):
        print("Downloading UCI HAR dataset...")
        urllib.request.urlretrieve(url, 'har.zip')
        import zipfile
        with zipfile.ZipFile('har.zip', 'r') as zip_ref:
            zip_ref.extractall('har')

    inner_zip = os.path.join('har', 'UCI HAR Dataset.zip')
    if os.path.exists(inner_zip) and not os.path.exists('UCI HAR Dataset'):
        import zipfile
        with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
            zip_ref.extractall('.')

download_har_if_needed()

def load_har_split(split='train'):
    base_dir = os.path.join('UCI HAR Dataset', split)
    signals_dir = os.path.join(base_dir, 'Inertial Signals')

    signal_files = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]

    channels = []
    for sig in signal_files:
        filepath = os.path.join(signals_dir, f"{sig}{split}.txt")
        data = np.loadtxt(filepath)
        channels.append(data)

    X = np.stack(channels, axis=-1).astype(np.float32)
    y_path = os.path.join(base_dir, f"y_{split}.txt")
    y = (np.loadtxt(y_path).astype(np.int64)) - 1
    sub_path = os.path.join(base_dir, f"subject_{split}.txt")
    subjects = np.loadtxt(sub_path).astype(np.int64)
    return X, y, subjects

def get_har_dataloaders(seed=0, batch_size=64):
    set_seed(seed)
    X_tr_raw, y_tr_raw, sub_tr = load_har_split('train')
    X_te_raw, y_te_raw, _ = load_har_split('test')

    unique_subs = np.unique(sub_tr)
    rng = np.random.default_rng(seed)
    val_subs = rng.choice(unique_subs, size=4, replace=False)

    val_mask = np.isin(sub_tr, val_subs)
    train_mask = ~val_mask

    X_tr, y_tr = X_tr_raw[train_mask], y_tr_raw[train_mask]
    X_val, y_val = X_tr_raw[val_mask], y_tr_raw[val_mask]
    X_te, y_te = X_te_raw, y_te_raw

    mean = X_tr.mean(axis=(0, 1), keepdims=True)
    std = X_tr.std(axis=(0, 1), keepdims=True) + 1e-8

    X_tr = (X_tr - mean) / std
    X_val = (X_val - mean) / std
    X_te = (X_te - mean) / std

    tr_loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te)), batch_size=batch_size, shuffle=False)
    return tr_loader, val_loader, te_loader, y_tr_raw

# ==========================================
# Task 2.1: Scaled Dot-Product Attention
# ==========================================
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V shapes: (batch_size, num_heads, seq_len, d_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

def test_attention():
    bs, heads, seq_len, d_k = 2, 4, 8, 16
    Q = torch.randn(bs, heads, seq_len, d_k)
    K = torch.randn(bs, heads, seq_len, d_k)
    V = torch.randn(bs, heads, seq_len, d_k)

    out, weights = scaled_dot_product_attention(Q, K, V)

    # Verify rows sum to 1
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5)

    # Check match with PyTorch functional scaled_dot_product_attention
    ref_out = F.scaled_dot_product_attention(Q, K, V)
    assert torch.allclose(out, ref_out, atol=1e-5)

    # Check causal mask enforces zero weight for future positions
    causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    _, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
    upper_tri = torch.triu(masked_weights[0, 0], diagonal=1)
    assert torch.allclose(upper_tri, torch.zeros_like(upper_tri), atol=1e-5)

test_attention()
print("Task 2.1: Scaled Dot-Product Attention Unit Tests Passed!")

# ==========================================
# Task 2.2: Multi-Head Attention & Encoder Block
# ==========================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        bs = Q.size(0)
        q = self.W_Q(Q).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_K(K).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_V(V).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_out, weights = scaled_dot_product_attention(q, k, v, mask)
        concat_out = attn_out.transpose(1, 2).contiguous().view(bs, -1, self.d_model)
        output = self.W_O(concat_out)
        return output, weights

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        norm_x = self.ln1(x)
        attn_out, weights = self.mha(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x, weights

def verify_parameter_counts():
    d_model, num_heads, d_ff = 128, 4, 512
    block = TransformerEncoderBlock(d_model, num_heads, d_ff, dropout=0.0)

    weights_only = 4 * (d_model * d_model) + 2 * (d_model * d_ff)
    total_params = sum(p.numel() for p in block.parameters())

    print(f"Hand-derived pure weights count: {weights_only}")
    print(f"Total block params (including biases & LayerNorms): {total_params}")
    assert weights_only == 196608

verify_parameter_counts()

# ==========================================
# Task 2.3: Task A - HAR Encoder Classifier & Training Loop
# ==========================================
class HAREncoderClassifier(nn.Module):
    def __init__(self, d_model=128, num_heads=4, d_ff=512, pos_type="sinusoidal"):
        super().__init__()
        self.pos_type = pos_type
        self.input_projection = nn.Linear(9, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        if pos_type == "sinusoidal":
            self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=200)
        elif pos_type == "learned":
            self.pos_enc = nn.Parameter(torch.randn(1, 200, d_model))
        elif pos_type == "none":
          self.pos_enc = None

        self.encoder = TransformerEncoderBlock(d_model, num_heads, d_ff)
        self.classifier = nn.Linear(d_model, 6)

    def forward(self, x):
        bs, seq_len, _ = x.shape
        x_emb = self.input_projection(x)
        cls_tokens = self.cls_token.expand(bs, -1, -1)
        x_emb = torch.cat((cls_tokens, x_emb), dim=1)

        if self.pos_type == "sinusoidal":
            x_emb = self.pos_enc(x_emb)
        elif self.pos_type == "learned":
            x_emb = x_emb + self.pos_enc[:, :seq_len+1]
        elif self.pos_type == "none":
            pass

        out, _ = self.encoder(x_emb)
        return self.classifier(out[:, 0])

# Train HAR Transformer for all 3 Positional Encoding variants
def train_har_transformer_all():
    pos_types = ["none", "sinusoidal", "learned"]
    acc_results = {}

    for pos_type in pos_types:
        accs, f1s = [], []
        for seed in [0, 1, 2]:
            set_seed(seed)
            tr_loader, val_loader, te_loader, _ = get_har_dataloaders(seed=seed)

            model = HAREncoderClassifier(d_model=128, num_heads=4, d_ff=512, pos_type=pos_type).to(device)
            optimizer = optim.Adam(model.parameters(), lr=1e-3)
            criterion = nn.CrossEntropyLoss()

            t0 = time.time()
            for epoch in range(15):
                model.train()
                for bx, by in tr_loader:
                    bx, by = bx.to(device), by.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(bx), by)
                    loss.backward()
                    optimizer.step()
            sec_per_epoch = (time.time() - t0) / 15.0

            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for bx, by in te_loader:
                    bx = bx.to(device)
                    preds = model(bx).argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_targets.extend(by.numpy())

            acc = np.mean(np.array(all_preds) == np.array(all_targets)) * 100.0
            f1 = f1_score(all_targets, all_preds, average='macro')
            accs.append(acc)
            f1s.append(f1)

            # Log individual run
            log_result({
                "problem": "P2_TaskA_HAR",
                "model": f"Transformer_{pos_type}",
                "seed": seed,
                "seq_len": 128,
                "test_acc": round(acc, 2),
                "macro_f1": round(f1, 4),
                "params": sum(p.numel() for p in model.parameters()),
                "sec_per_epoch": round(sec_per_epoch, 3),
                "notes": f"HAR Transformer Positional Encoding: {pos_type}"
            })

        mean_acc = np.mean(accs)
        acc_results[pos_type] = mean_acc
        print(f"HAR Transformer ({pos_type}): Mean Acc = {mean_acc:.2f}%, Mean Macro-F1 = {np.mean(f1s):.4f}")

    # Plot & Save positional_encoding_ablation.png
    plt.figure(figsize=(7, 4))
    plt.bar(['None', 'Sinusoidal', 'Learned'], [acc_results['none'], acc_results['sinusoidal'], acc_results['learned']], color=['gray', 'teal', 'darkblue'])
    plt.ylabel('Test Accuracy (%)')
    plt.title('Task A: Positional Encoding Ablation on HAR')
    plt.grid(axis='y', linestyle='--')
    plt.savefig(os.path.join(OS_FIGURES_DIR, "positional_encoding_ablation.png"), bbox_inches='tight')
    plt.close()

train_har_transformer_all()

# ==========================================
# Task 2.4: Task B - Character Language Model
# ==========================================
def download_tinyshakespeare():
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    if not os.path.exists('tinyshakespeare.txt'):
        print("Downloading Tiny Shakespeare dataset...")
        urllib.request.urlretrieve(url, 'tinyshakespeare.txt')
    with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
        text = f.read()
    return text

text = download_tinyshakespeare()
chars = sorted(list(set(text)))
vocab_size = len(chars)
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}
data = np.array([char2idx[ch] for ch in text], dtype=np.int64)

Attention Unit Tests Passed!
Encoder Parameter Derivation Verified Successfully!
Problem 2 Execution & Generation Completed Successfully!
